In [1]:
import pickle
from nltk.corpus import wordnet as wn
import pandas as pd

In [2]:
def open_list(l):
    out = []
    for ll in l:
        out.extend(ll)
    return out

def find_cohyponyms(synset):
    cohyponym_synsets = []
    cohyponym_lemmas = []

    pre_cohyponyms = [hprn.hyponyms() for hprn in synset.hypernyms()]
    for cohyps in pre_cohyponyms: 
        pre_cohyponym_lemmas = [chpl for chpl in [cohyp.lemma_names() for cohyp in cohyps]]
        for pcl in pre_cohyponym_lemmas:
            cohyponym_lemmas.extend(pcl)
        cohyponym_synsets.extend([cohyp.name() for cohyp in cohyps])

    return cohyponym_synsets, cohyponym_lemmas

In [3]:
def process_all_children(synset, dicts2store, lemma=None, level=0):
    synset_dict = dict()
    synset_dict['subset'] = 'appendix'

    synset_dict['wordnet_id'] = synset.offset()
    synset_dict['core_synset'] = synset.name()

    synset_dict['core_lemma'] = lemma.strip('\n')

    synset_dict['hypernym_synsets'] = [h.name() for h in synset.hypernyms()]
    #synset_dict['hypernym_lemmas'] = [h.lemma_names()[0] for h in synset.hypernyms()]

    #synset_dict['cohyponym_synsets'], synset_dict['cohyponym_lemmas'] = find_cohyponyms(synset)
    synset_dict['cohyponym_synsets'] = find_cohyponyms(synset)[0]
    dicts2store.append(synset_dict)

    if level < 1:
        for child in synset.hyponyms():
            dicts2store.extend(process_all_children(child, [], child.lemma_names()[0].strip('\n'), level+1))    
    
    return dicts2store

In [ ]:
app_data_dicts = []

with open('article_20_appendixA.txt', 'r', encoding='utf-8') as f:
    for line in f.readlines():
        synset_name, lemma = line.split(' ')[:2]
        synset = wn.synset(synset_name)
        synset_dicts = process_all_children(synset, [], lemma)
        app_data_dicts.extend(synset_dicts)

In [27]:
with open('appendix_core_lemmas.txt', 'w', encoding='utf-8') as f:
    for d in app_data_dicts:
        f.write(d['core_lemma'].strip('\n')+'\n')

In [28]:
with open('appendix_predicted_hypernyms.txt', 'r', encoding='utf-8') as f:
    for pred_line, data_dict in zip(f.readlines(), app_data_dicts):
        data_dict['pred_hypernym'] = pred_line.strip('\n')

In [ ]:
taxo_data_dicts = []

with open('taxollama3_test.pickle', 'rb') as f:
    taxo3 = pickle.load(f)

for node in taxo3:
    node_dict = dict()

    if node['case'] == 'leafs_and_no_leafs' or node['case'] == 'simple_triplet_2parent':
        if type(node['children']) is str:
            synset = wn.synset(node['children'])
        elif type(node['children']) is list:
            synset = wn.synset(node['children'][0])

        hypernyms = []
        if type(node['parents']) is str:
            hypernyms.append(node['parents'])
        elif type(node['parents']) is list:
            hypernyms = node['parents']
        node_dict['hypernym_synsets'] = hypernyms
        #node_dict['hypernym_lemmas'] = open_list([wn.synset(h).lemma_names() for h in node_dict['hypernym_synsets']])

    
    elif node['case'] == 'predict_hypernym':
        synset = wn.synset(node['parents'])
        node_dict['hypernym_synsets'] = [h.name() for h in synset.hypernyms()]
        #node_dict['hypernym_lemmas'] = open_list([h.lemma_names() for h in synset.hypernyms()])

    node_dict['subset'] = node['case']
    node_dict['wordnet_id'] = synset.offset()
    node_dict['core_synset'] = synset.name()
    node_dict['core_lemma'] = synset.lemma_names()[0].strip('\n')

    node_dict['cohyponym_synsets'], _ = find_cohyponyms(synset)
    node_dict['cohyponym_synsets'].remove(node_dict['core_synset'])
    #node_dict['cohyponym_synsets'], node_dict['cohyponym_lemmas'] = find_cohyponyms(synset)

    taxo_data_dicts.append(node_dict)
        

In [30]:
# adding model's predictions
with open('_VityaVitalich-Llama3.1-8b-instructqlora_r16_lr_3e-5_bs32_0', 'rb') as f:
    taxo3_preds = pickle.load(f)

predictions = []

for pred in taxo3_preds:
    if ',' in pred[0]:
        the_pred = pred[0].split(',')[0].replace('hypernym:', '').strip()
        if not the_pred.replace(' ', '').replace('-', '').replace('\'', '').isalpha() and len(the_pred) < 50:
            the_pred = pred[1].split(',')[0].replace('hypernym:', '').strip()
    else:
        the_pred = pred[1].split(',')[0].replace('hypernym:', '').strip()
    predictions.append(the_pred)

In [31]:
for pred_line, data_dict in zip(predictions, taxo_data_dicts):
    data_dict['pred_hypernym'] = pred_line

In [32]:
final_df = pd.DataFrame.from_dict(app_data_dicts + taxo_data_dicts)
final_df.to_csv('taxo2img_test_set_for_DiffMs.tsv', sep='\t')

In [33]:
final_df.head(3)

,subset,wordnet_id,core_synset,core_lemma,hypernym_synsets,cohyponym_synsets,pred_hypernym
0,appendix,13388245,coin.n.01,coin,[coinage.n.01],[coin.n.01],numismatics
1,appendix,13389105,bawbee.n.01,bawbee,[coin.n.01],"[bawbee.n.01, bezant.n.01, change.n.08, crown....",bee
2,appendix,13389194,bezant.n.01,bezant,[coin.n.01],"[bawbee.n.01, bezant.n.01, change.n.08, crown....",gold coin


### Reformatting

In [34]:
import hashlib
wordnet_ids = set(final_df['wordnet_id'])

def hash_lemma(lemma):
    new_id = int(hashlib.sha256(lemma.encode('utf-8')).hexdigest(), 16) % 10**8
    if new_id not in wordnet_ids:
        wordnet_ids.add(new_id)
        return new_id
    else:
        return hash_lemma(lemma+'_meow')

In [35]:
preds_data_dicts = []
for row in final_df.iterrows():
    pred_data_dict = {
        'core_lemma': row[1]['pred_hypernym'],
        'subset': row[1]['subset'] + '_llama',
        'wordnet_id': hash_lemma(row[1]['pred_hypernym'])
    }
    if len(row[1]['hypernym_synsets']) == 1:
        pred_data_dict['core_synset'] = row[1]['hypernym_synsets'][0]
        pred_synset = wn.synset(pred_data_dict['core_synset'])
    elif len(row[1]['hypernym_synsets']) > 1:
        pred_data_dict['core_synset'] = row[1]['hypernym_synsets'][0]
        for hpr in row[1]['hypernym_synsets']:
            if pred_data_dict['core_lemma'] in wn.synset(hpr).lemma_names():
                pred_data_dict['core_synset'] = hpr
                break
        pred_synset = wn.synset(pred_data_dict['core_synset'])
    else:
        print(pred_data_dict['core_lemma'])
        pred_data_dict['cohyponym_synsets'] = None
        pred_data_dict['hypernym_synsets'] = None
        continue
        
    pred_data_dict['hypernym_synsets'] = [h.name() for h in pred_synset.hypernyms()]
    if len(pred_data_dict['hypernym_synsets']) > 0:
        pred_data_dict['cohyponym_synsets'] = find_cohyponyms(pred_synset)[0]
        pred_data_dict['cohyponym_synsets'].remove(pred_data_dict['core_synset'])
    else:
        pred_data_dict['cohyponym_synsets'] = []
    
    preds_data_dicts.append(pred_data_dict)
    

In [36]:
pred_df = pd.DataFrame().from_dict(preds_data_dicts)
full_df = pd.concat([final_df.drop('pred_hypernym', axis=1), pred_df])

In [37]:
full_df
# в _llama если есть core_synset, то это тот, который верный для ребёнка (родителя для leaf no leaf), по которому предсказано

,subset,wordnet_id,core_synset,core_lemma,hypernym_synsets,cohyponym_synsets
0,appendix,13388245,coin.n.01,coin,[coinage.n.01],[coin.n.01]
1,appendix,13389105,bawbee.n.01,bawbee,[coin.n.01],"[bawbee.n.01, bezant.n.01, change.n.08, crown...."
2,appendix,13389194,bezant.n.01,bezant,[coin.n.01],"[bawbee.n.01, bezant.n.01, change.n.08, crown...."
3,appendix,13388111,change.n.08,change,[coin.n.01],"[bawbee.n.01, bezant.n.01, change.n.08, crown...."
4,appendix,13389864,crown.n.06,crown,[coin.n.01],"[bawbee.n.01, bezant.n.01, change.n.08, crown...."
...,...,...,...,...,...,...
1680,leafs_and_no_leafs_llama,41796839,area.n.03,renal area,[body_part.n.01],"[abdomen.n.01, adnexa.n.01, ambulacrum.n.01, a..."
1681,predict_hypernym_llama,36389327,card_player.n.01,bridge player,[player.n.01],"[ballplayer.n.01, billiard_player.n.01, bowler..."
1682,predict_hypernym_llama,27347584,domestic_partner.n.01,married woman,[person.n.01],"[abator.n.01, abjurer.n.01, abomination.n.01, ..."
1683,predict_hypernym_llama,13967420,body_part.n.01,abdomen,[part.n.03],"[acicula.n.01, base.n.04, corner.n.09, corpus...."


# Adding images

### Different parsing approach

In [38]:
import requests
from bs4 import BeautifulSoup
from fake_useragent import UserAgent
import re
from tqdm import tqdm
import json
import urllib.request

Ищем такое:


<div class="sdms-search-results__list sdms-search-results__list--image"><a class="sdms-image-result" href="https://commons.wikimedia.org/wiki/File:Monkey_eating.jpg" title="File:Monkey eating.jpg" style="width: 246px;"><img data-src="https://upload.wikimedia.org/wikipedia/commons/thumb/c/c8/Monkey_eating.jpg/250px-Monkey_eating.jpg" alt="Monkey eating.jpg" class="sd-image" loading="lazy" style="height: 100% !important; max-width: 1713px !important; max-height: 1255px;" src="https://upload.wikimedia.org/wikipedia/commons/thumb/c/c8/Monkey_eating.jpg/250px-Monkey_eating.jpg"></a>

https://commons.wikimedia.org/wiki/File:Monkey_eating.jpg -> https://upload.wikimedia.org/wikipedia/commons/thumb/c/c8/Monkey_eating.jpg/800px-Monkey_eating.jpg?20090816085702

<img alt="File:Monkey eating.jpg" src="https://upload.wikimedia.org/wikipedia/commons/thumb/c/c8/Monkey_eating.jpg/800px-Monkey_eating.jpg?20090816085702" decoding="async" width="800" height="586" srcset="https://upload.wikimedia.org/wikipedia/commons/thumb/c/c8/Monkey_eating.jpg/1200px-Monkey_eating.jpg?20090816085702 1.5x, https://upload.wikimedia.org/wikipedia/commons/thumb/c/c8/Monkey_eating.jpg/1600px-Monkey_eating.jpg?20090816085702 2x" data-file-width="1713" data-file-height="1255">

In [45]:
def download_image(url, save_as):
    urllib.request.urlretrieve(url, save_as)

In [59]:
pages = dict()
img_path_column = []
all_image_hrefs = []

link = re.compile('(?<=href=\").+?(?=\")')
img_link = re.compile('(?<=src=\").+?(?=\")')
alt_re = re.compile('(?<=alt=\").+?(?=\")')

for row in tqdm(full_df.iterrows(), total=len(full_df)):
    concept = row[1]['core_lemma']
    if concept not in pages.keys():
        url = f"https://commons.wikimedia.org/w/index.php?search={concept}&title=Special:MediaSearch&go=Go&type=image"
        user_agent = UserAgent().chrome  # хотим притворяться браузером
        response = requests.get(url, headers={'User-Agent':user_agent})
        soup = BeautifulSoup(response.text, 'html.parser')
        posts = soup.find_all('a', {'class': 'sdms-image-result'}) # вынет ли оно картинку, которая там внутри?

        image_downloaded = False
        found_images = []
        img_path = None

        for post in posts[:5]:
            real_page_url = re.search(link, str(post))
            alt = re.search(alt_re, str(post))
            if alt is None or real_page_url is None:
                continue
            else:
                real_page_url = real_page_url[0]
                alt = alt[0]
            response2 = requests.get(real_page_url, headers={'User-Agent':user_agent})
            soup2 = BeautifulSoup(response2.text, 'html.parser')
            img = soup2.find('img', {'alt':'File:'+alt})
            link_to_img = re.search(img_link, str(img))
            if link_to_img is None:
                continue
            else:
                link_to_img = link_to_img[0]
            found_images.append(link_to_img)

            if not image_downloaded:
                try:
                    download_image(link_to_img, 'wikiCommonsOutput/'+alt)
                    image_downloaded = True
                    img_path = alt
                    img_path_column.append(img_path)
                except:
                    continue
        
        all_image_hrefs.append(found_images)
        pages[concept] = [found_images, img_path]

        with open('parsed_wikimedia_imgs_2.json', 'w', encoding='utf-8') as f:
            json.dump(pages, f, indent=3, ensure_ascii=False)

    else:
        found_images, img_path = pages[concept]
        img_path_column.append(img_path)
        all_image_hrefs.append(found_images)

  0%|          | 0/3370 [00:00<?, ?it/s]

100%|██████████| 3370/3370 [3:25:57<00:00,  3.67s/it]  


In [62]:
img_path_column = []
all_image_hrefs = []

for row in full_df.iterrows():
    res = pages.get(row[1]['core_lemma'])
    if res is None:
        ipc, aih = None, None
    else:
        aih, ipc = res[0], res[1]
    img_path_column.append(ipc)
    all_image_hrefs.append(aih)

full_df['wikimedia_images'] = all_image_hrefs
full_df['image_filename'] = img_path_column
full_df.to_csv('taxo2img_test_set_for_DiffMs_with_images_paths_2.tsv', sep='\t', index=False)

In [64]:
len(full_df[full_df['image_filename'].isna()])

32